In [3]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def calculate_gradient (theta,X,y):
    m = y.size 
    return (X.T @ (sigmoid(X@theta) - y)) / m 

def gradient_descent(X,y,alpha=0.1,num_iter=100, tol = 1e-7):
    X_b = np.c_[np.ones((X.shape[0], 1)), X] 
    theta = np.zeros(X_b.shape[1])

    for i in range(num_iter):
        grad = calculate_gradient(theta, X_b,y)
        theta -= alpha*grad

        if np.linalg.norm(grad)<tol:
            break 
    
    return theta
    
def predict_proba(X,theta):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return sigmoid(X_b @ theta)

def predict(X, theta, threshold = 0.5):
    return(predict_proba(X,theta )>= threshold).astype(int)

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("framingham.csv")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score 

X = df.drop("TenYearCHD", axis=1)
y = df["TenYearCHD"]

In [5]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_imputed,y,test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

theta_hat = gradient_descent(X_train_scaled, y_train.values, alpha=0.1)

y_pred_train = predict(X_train_scaled, theta_hat)
y_pred_test = predict(X_test_scaled, theta_hat)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(train_acc)
print(test_acc)

0.8525073746312685
0.8679245283018868


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred)
    }

from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X_train_scaled, y_train)

y_pred_sk = sk_model.predict(X_test_scaled)

sk_metrics = evaluate(y_test, y_pred_sk)
print(sk_metrics)


theta_hat = gradient_descent(
    X_train_scaled, 
    y_train.values, 
    alpha=0.1, 
    num_iter=1000
)
y_pred_scratch = predict(X_test_scaled, theta_hat)
scratch_metrics = evaluate(y_test, y_pred_scratch)
print(scratch_metrics)

print(y.value_counts())
print(y.value_counts(normalize=True))

{'Accuracy': 0.8667452830188679, 'Precision': 0.6666666666666666, 'Recall': 0.06837606837606838, 'F1-score': 0.12403100775193798}
{'Accuracy': 0.8667452830188679, 'Precision': 0.6666666666666666, 'Recall': 0.06837606837606838, 'F1-score': 0.12403100775193798}
TenYearCHD
0    3594
1     644
Name: count, dtype: int64
TenYearCHD
0    0.848042
1    0.151958
Name: proportion, dtype: float64
